# Supplementary Figure 6

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(gridGraphics)
  library(patchwork)
  library(scales)
})

set.seed(20260803)

figure_dir <- "."
data_dir <- file.path(figure_dir, "data")
data_paths <- file.path(data_dir, c(
  "Supplementary_Figure6a.rds",
  "Supplementary_Figure6b.rds",
  "Supplementary_Figure6c.rds"
))
table_s3_path <- file.path(
  dirname(dirname(figure_dir)), "Supplementary_Tables", "TableS5.csv"
)
stopifnot(
  all(file.exists(data_paths)), all(file.info(data_paths)$size > 0),
  file.exists(table_s3_path), file.info(table_s3_path)$size > 0
)

s6a_data <- readRDS(data_paths[1])
s6b_data <- readRDS(data_paths[2])
s6c_data <- readRDS(data_paths[3])
table_s3 <- fread(table_s3_path)
table_s3_cos <- sort(unique(unlist(strsplit(
  na.omit(table_s3[["Robust CoS ID"]]), "; ", fixed = TRUE
))))
table_s3_cos <- table_s3_cos[nzchar(table_s3_cos) & table_s3_cos != "NA"]
table_s3_ad_gp <- table_s3[
  get("AD colocalized") == "Yes", get("Glycopeptiform ID")
]

stopifnot(
  identical(s6a_data$section, "S6a"),
  identical(s6b_data$section, "S6b"),
  identical(s6c_data$section, "S6c"),
  identical(as.integer(s6c_data$region$x_range), c(44200000L, 44600000L)),
  identical(s6c_data$y_limits_source, "plot_initial defaults; no panel-specific overrides"),
  identical(vapply(s6c_data$cos_sets, `[[`, character(1), "cos_id"), table_s3_cos),
  setequal(
    s6b_data$traits$trait_id[s6b_data$traits$modality == "Glycopeptiform"],
    table_s3_ad_gp
  ),
  ncol(s6c_data$log10p) == 40,
  length(s6c_data$positions) == nrow(s6c_data$log10p)
)

glycan_style <- data.table(
  component = c("HexNAc", "Hex", "Fuc", "NeuAc"),
  pch = c(22, 21, 24, 23),
  fill = c("#2F7ED8", "#68C83E", "#D84A4A", "#9667CF"),
  border = c("#1F4E79", "#2E6B25", "#7F1D1D", "#5B3A8A")
)

parse_glycan <- function(composition) {
  if (identical(composition, "Hex")) {
    parsed <- data.table(component = "Hex", count = 1L)
  } else {
    hits <- regmatches(
      composition,
      gregexpr("(HexNAc|Hex|Fuc|NeuAc)\\(([0-9]+)\\)", composition, perl = TRUE)
    )[[1]]
    stopifnot(length(hits) > 0)
    parsed <- rbindlist(lapply(hits, function(hit) {
      fields <- regmatches(
        hit,
        regexec("(HexNAc|Hex|Fuc|NeuAc)\\(([0-9]+)\\)", hit, perl = TRUE)
      )[[1]]
      data.table(component = fields[2], count = as.integer(fields[3]))
    }))
  }
  merge(parsed, glycan_style, by = "component", sort = FALSE)
}



## Panels a-b: genotype-stratified molecular abundance

In [ ]:
make_panel_label_grob <- function(line1, line2 = NULL, glycan = NULL) {
  children <- list(grid::textGrob(
    line1,
    x = grid::unit(0.97, "npc"), y = grid::unit(0.96, "npc"),
    just = c("right", "top"),
    gp = grid::gpar(fontsize = 11.5, fontface = "bold")
  ))
  if (!is.null(line2)) {
    children <- c(children, list(grid::textGrob(
      line2,
      x = grid::unit(0.97, "npc"), y = grid::unit(0.90, "npc"),
      just = c("right", "top"),
      gp = grid::gpar(fontsize = 11)
    )))
  }
  if (!is.null(glycan)) {
    symbol_x <- 0.94 - rev(seq_len(nrow(glycan)) - 1) * 0.09
    for (ii in seq_len(nrow(glycan))) {
      children <- c(children, list(
        grid::textGrob(
          paste0(glycan$count[ii], "×"),
          x = grid::unit(symbol_x[ii] - 0.018, "npc"), y = grid::unit(0.87, "npc"),
          just = c("right", "centre"), gp = grid::gpar(fontsize = 11.2)
        ),
        grid::pointsGrob(
          x = grid::unit(symbol_x[ii], "npc"), y = grid::unit(0.87, "npc"),
          pch = glycan$pch[ii], size = grid::unit(3.0, "mm"),
          gp = grid::gpar(fill = glycan$fill[ii], col = glycan$border[ii], lwd = 0.8)
        )
      ))
    }
  }
  do.call(grid::grobTree, children)
}

make_genotype_panel <- function(panel_df, label_grob, variant, genotype_levels,
                                genotype_colors, tag = NULL) {
  panel_df <- as.data.table(panel_df)
  panel_df[, genotype := factor(genotype, levels = genotype_levels)]
  stopifnot(uniqueN(panel_df$genotype) == length(genotype_levels), !anyNA(panel_df$value))
  panel_med <- panel_df[, .(median_y = median(value)), by = genotype]
  panel_range <- range(panel_df$value)
  panel_span <- diff(panel_range)
  if (!is.finite(panel_span) || panel_span <= 0) panel_span <- 1
  panel_limits <- c(panel_range[1] - 0.05 * panel_span, panel_range[2] + 0.25 * panel_span)
  panel_breaks <- scales::breaks_pretty(n = 4)(panel_range)
  panel_breaks <- panel_breaks[panel_breaks >= panel_limits[1] & panel_breaks <= panel_range[2]]

  ggplot(panel_df, aes(x = genotype, y = value)) +
    geom_boxplot(width = 0.5, alpha = 0.7, outlier.shape = NA, color = "black") +
    geom_jitter(width = 0.2, size = 2, alpha = 0.6, aes(color = genotype)) +
    geom_line(
      data = panel_med,
      aes(x = as.numeric(genotype), y = median_y, group = 1),
      inherit.aes = FALSE,
      color = "grey50", linewidth = 1, linetype = "dashed"
    ) +
    scale_color_manual(values = genotype_colors) +
    scale_y_continuous(limits = panel_limits, breaks = panel_breaks, expand = c(0, 0)) +
    annotation_custom(label_grob, xmin = -Inf, xmax = Inf, ymin = -Inf, ymax = Inf) +
    labs(x = paste0("Genotype (", variant, ")"), y = "Normalized abundance", tag = tag) +
    theme_bw(base_size = 14) +
    theme(
      axis.text.x = element_text(size = 14, face = "bold"),
      axis.text.y = element_text(size = 14),
      axis.title = element_text(size = 16, face = "bold"),
      axis.title.y = element_text(margin = margin(r = 5)),
      legend.position = "none",
      panel.grid = element_blank(),
      panel.border = element_rect(linewidth = 1.2),
      plot.tag = element_text(size = 22, face = "bold", hjust = 0, vjust = 1),
      plot.tag.position = c(0, 1.03),
      plot.margin = margin(t = 4, r = 9, b = 4, l = 9)
    )
}

panel_a <- make_genotype_panel(
  s6a_data$plot_df,
  make_panel_label_grob(s6a_data$trait$gene, line2 = s6a_data$trait$modality),
  variant = s6a_data$variant$id,
  genotype_levels = s6a_data$genotype_levels,
  genotype_colors = s6a_data$genotype_colors,
  tag = "a"
)

panel_b_df <- as.data.table(s6b_data$plot_df)
trait_metadata <- as.data.table(s6b_data$traits)[order(display_order)]
panel_b_plots <- vector("list", nrow(trait_metadata))
for (ii in seq_len(nrow(trait_metadata))) {
  trait <- trait_metadata[ii]
  is_glycan <- identical(trait$modality, "Glycopeptiform")
  panel_b_plots[[ii]] <- make_genotype_panel(
    panel_b_df[trait_id == trait$trait_id],
    make_panel_label_grob(
      paste(trait$gene, if (is_glycan) trait$site else ""),
      line2 = if (!is_glycan) trait$modality else NULL,
      glycan = if (is_glycan) parse_glycan(trait$glycan_composition) else NULL
    ),
    variant = s6b_data$variant$id,
    genotype_levels = s6b_data$genotype_levels,
    genotype_colors = s6b_data$genotype_colors,
    tag = if (ii == 1L) "b" else NULL
  )
}

top_grid <- wrap_plots(c(list(panel_a), panel_b_plots), ncol = 5, nrow = 2) +
  plot_layout(axis_titles = "collect_y")



## Panel c: regional colocalization tracks

In [ ]:
as_coloc_glyphs <- function(composition) {
  parsed <- parse_glycan(composition)
  lapply(seq_len(nrow(parsed)), function(ii) list(
    count = parsed$count[ii],
    pch = parsed$pch[ii],
    fill = parsed$fill[ii],
    border = parsed$border[ii]
  ))
}

draw_outcome_annotation <- function(annotation, annotation_scale = 1) {
  usr <- par("usr")
  x_span <- usr[2] - usr[1]
  y_span <- usr[4] - usr[3]
  x_left <- usr[1] + 0.035 * x_span
  y_top <- usr[4] - 0.035 * y_span
  halo_dx <- 0.0012 * x_span
  halo_dy <- 0.0012 * y_span
  halo_offsets <- expand.grid(dx = c(-halo_dx, 0, halo_dx), dy = c(-halo_dy, 0, halo_dy))
  halo_offsets <- halo_offsets[halo_offsets$dx != 0 | halo_offsets$dy != 0, ]

  draw_halo_text <- function(x, y, labels, adj = c(0, 1), cex = 1, font = 1) {
    for (jj in seq_len(nrow(halo_offsets))) {
      text(
        x + halo_offsets$dx[jj], y + halo_offsets$dy[jj], labels = labels,
        adj = adj, cex = cex, font = font, col = "white", xpd = FALSE
      )
    }
    text(x, y, labels = labels, adj = adj, cex = cex, font = font, col = "black", xpd = FALSE)
  }

  draw_halo_text(x_left, y_top, annotation$line1, cex = 1.38 * annotation_scale, font = 2)
  if (!is.null(annotation$line2)) {
    draw_halo_text(
      x_left, y_top - 0.095 * y_span, annotation$line2,
      cex = 1.28 * annotation_scale
    )
  }
  if (!is.null(annotation$glycan)) {
    glycan_y <- y_top - 0.095 * y_span
    for (ii in seq_along(annotation$glycan)) {
      component <- annotation$glycan[[ii]]
      component_x <- x_left + (ii - 1) * 0.092 * x_span
      draw_halo_text(
        component_x, glycan_y, paste0(component$count, "×"),
        adj = c(0, 0.5), cex = 1.28 * annotation_scale
      )
      points(
        component_x + 0.048 * x_span, glycan_y,
        pch = component$pch, bg = "white", col = "white",
        cex = 2.02 * annotation_scale, lwd = 1.2, xpd = FALSE
      )
      points(
        component_x + 0.048 * x_span, glycan_y,
        pch = component$pch, bg = component$fill, col = component$border,
        cex = 1.82 * annotation_scale, lwd = 1.0, xpd = FALSE
      )
    }
  }
}

draw_grn_track <- function(x_range, tick_at, right_margin) {
  par(mar = c(3.8, 5, 0.2, right_margin))
  plot.new()
  plot.window(xlim = x_range, ylim = c(-0.35, 0.35), xaxs = "i", yaxs = "i")
  with(s6c_data$gene$introns, arrows(
    start, 0, end, 0, length = 0.045, angle = 25, code = 2, lwd = 0.7
  ))
  with(s6c_data$gene$exons, rect(
    start, -0.18, end, 0.18, col = "steelblue", border = NA
  ))
  axis(
    1,
    at = tick_at,
    labels = paste0(formatC(tick_at / 1e6, format = "f", digits = 2), "M"),
    cex.axis = 2
  )
  mtext(s6c_data$gene$name, side = 1, line = 2.35, cex = 1.25, font = 2)
  box(lwd = 0.8)
}

draw_s6c <- function() {
  positions <- s6c_data$positions
  x_range <- s6c_data$region$x_range
  outcomes <- as.data.table(s6c_data$outcomes)[order(display_index)]
  y_limits <- as.data.table(s6c_data$y_limits)[order(display_index)]
  coloc_markers <- as.data.table(s6c_data$cos_markers)
  uncoloc_markers <- as.data.table(s6c_data$uncoloc_markers)
  plot_cols <- s6c_data$plot_cols
  plot_rows <- ceiling(nrow(outcomes) / plot_cols)
  stopifnot(plot_rows * plot_cols == nrow(outcomes))

  oldpar <- par(no.readonly = TRUE)
  on.exit(par(oldpar))
  layout(
    matrix(seq_len((plot_rows + 1L) * plot_cols), nrow = plot_rows + 1L, byrow = TRUE),
    heights = c(rep(1, plot_rows), 0.38),
    widths = rep(1, plot_cols)
  )
  par(oma = c(0, 0, 1, 0))

  for (ii in seq_len(nrow(outcomes))) {
    plot_col <- (ii - 1L) %% plot_cols + 1L
    right_margin <- if (plot_col < plot_cols) 1 else 1
    par(mar = c(0.4, 5, 0.8, right_margin))
    y <- s6c_data$log10p[, ii]
    plot(
      positions, y,
      pch = 16, col = s6c_data$background_color,
      xlab = "", ylab = expression(-log[10](P)), xaxt = "n",
      xlim = x_range, xaxs = "i",
      ylim = c(y_limits$ymin[ii], y_limits$ymax[ii]),
      cex.axis = 2, cex.lab = 2, font.lab = 1
    )

    panel_uncoloc <- uncoloc_markers[display_index == ii]
    if (nrow(panel_uncoloc)) {
      uncoloc_y <- y[match(panel_uncoloc$position, positions)]
      points(
        panel_uncoloc$position, uncoloc_y,
        pch = 4,
        col = grDevices::adjustcolor(panel_uncoloc$source_color, alpha.f = 0.68),
        cex = 1.5, lwd = 1.5
      )
      legend_rows <- unique(panel_uncoloc[, .(cos_id, source_color)])
      usr <- par("usr")
      legend(
        x = usr[1] + 0.035 * diff(usr[1:2]),
        y = usr[3] + 0.32 * diff(usr[3:4]),
        legend = rep("Colocalization not supported", nrow(legend_rows)),   # same wording as Figure 4c
        pch = 4,
        col = legend_rows$source_color,
        text.col = legend_rows$source_color,
        bty = "n", cex = 0.98, pt.cex = 1.15,
        x.intersp = 0.30, y.intersp = 0.92
      )
    }

    panel_coloc <- coloc_markers[display_index == ii]
    if (nrow(panel_coloc)) {
      for (this_cos in unique(panel_coloc$cos_id)) {
        cos_points <- panel_coloc[cos_id == this_cos]
        marker_y <- y[match(cos_points$position, positions)]
        marker_y[!is.finite(marker_y)] <- 0.96 * y_limits$ymax[ii]
        is_ad_lead <- this_cos == "cos2" &
          cos_points$position == s6c_data$lead_variant_position
        if (any(!is_ad_lead)) {
          points(
            cos_points$position[!is_ad_lead], marker_y[!is_ad_lead],
            pch = 21, bg = cos_points$source_color[!is_ad_lead],
            col = cos_points$source_color[!is_ad_lead], cex = 2.50, lwd = 1.6
          )
        }
        if (any(is_ad_lead)) {
          points(
            cos_points$position[is_ad_lead], marker_y[is_ad_lead],
            pch = 21, bg = "white", col = "white", cex = 3.25, lwd = 3.8
          )
          points(
            cos_points$position[is_ad_lead], marker_y[is_ad_lead],
            pch = 21, bg = "#F46D43", col = "#B2182B", cex = 2.85, lwd = 2.6
          )
        }
        label_i <- which.max(marker_y)
        label_pos <- if (marker_y[label_i] > 0.78 * y_limits$ymax[ii]) 1 else 3
        label_color <- if (is_ad_lead[label_i]) {
          "#B2182B"
        } else {
          cos_points$source_color[label_i]
        }
        text(
          cos_points$position[label_i], marker_y[label_i], labels = this_cos,
          pos = label_pos, offset = 0.72, cex = 1.02, font = 2, col = label_color
        )
      }
    }

    annotation <- list(line1 = outcomes$line1[ii])
    if (!is.na(outcomes$line2[ii])) annotation$line2 <- outcomes$line2[ii]
    if (!is.na(outcomes$glycan_composition[ii])) {
      annotation$glycan <- as_coloc_glyphs(outcomes$glycan_composition[ii])
    }
    draw_outcome_annotation(annotation)
  }

  for (track_col in seq_len(plot_cols)) {
    draw_grn_track(
      x_range,
      if (track_col < plot_cols) s6c_data$genomic_ticks else head(s6c_data$genomic_ticks, -1),
      right_margin = if (track_col < plot_cols) 1 else 1
    )
  }
  invisible()
}

outcomes_check <- as.data.table(s6c_data$outcomes)
glyco_rows <- which(!is.na(outcomes_check$duplicate_rank))
stopifnot(
  identical(s6c_data$schema_version, "1.3"),
  identical(
    vapply(s6c_data$cos_sets, `[[`, character(1), "cos_id"),
    c("cos2", "cos3")
  ),
  length(unique(vapply(s6c_data$cos_sets, `[[`, character(1), "color"))) == 2L,
  all(grepl("\\*$", outcomes_check$line1[glyco_rows]) ==
        (outcomes_check$duplicate_rank[glyco_rows] > 1L)),
  !any(grepl("\\b(AC|PCC|DLPFC|eQTL)\\b", paste(outcomes_check$line1, outcomes_check$line2))),
  nrow(s6c_data$cos_markers) == 17L,
  nrow(s6c_data$uncoloc_markers) == 143L
)

s6c <- patchwork::wrap_elements(
  full = gridGraphics::echoGrob(draw_s6c, prefix = "supplementary_figure6c")
) +
  labs(tag = "c") +
  theme(
    plot.tag = element_text(size = 22, face = "bold", hjust = 0, vjust = 1),
    plot.tag.position = c(0, 1.01)
  )


## Assembly and vector PDF export

In [ ]:
supplementary_figure6 <- top_grid / s6c +
  plot_layout(heights = c(8.3, 19.2)) +
  plot_annotation(
    caption = "* Every glycopeptiform after the first identical APOD site/composition occurrence. Robust CoS after p-value < 1e-4 filtering are shown.",
    theme = theme(
      plot.caption = element_text(size = 11, hjust = 0, color = "black", margin = margin(t = 4)),
      plot.margin = margin(5, 6, 5, 6)
    )
  )

suppressWarnings(ggsave(
  filename = file.path(figure_dir, "Supplementary_Figure6.pdf"),
  plot = supplementary_figure6,
  device = grDevices::pdf,
  width = 28,
  height = 28,
  units = "in",
  limitsize = FALSE,
  bg = "white"
))

message("Saved from minimal section RDS files: ", file.path(figure_dir, "Supplementary_Figure6.pdf"))